In [1]:
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi

In [3]:
import importlib
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.data_cleaning.data import _get_project_data_dir, download_data, load_data, clean_data


In [2]:
download_data()

Dataset déjà téléchargé, téléchargement ignoré.


In [4]:
df = load_data()
df_cleaned = clean_data(df)

KeyboardInterrupt: 

In [4]:
# pip install nltk si besoin
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [9]:
_stop_words = set(stopwords.words('english'))
_lemmatizer = WordNetLemmatizer()

def preprocess_text(text, remove_stopwords=True, remove_punctuation=True, lemmatize=True):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    
    if remove_punctuation:
        tokens = [t for t in tokens if t.isalpha()]
    
    if remove_stopwords:
        tokens = [t for t in tokens if t not in _stop_words]
    
    if lemmatize:
        for pos in ('n', 'v', 'a'):
            tokens = [_lemmatizer.lemmatize(t, pos=pos) for t in tokens]
    
    return ' '.join(tokens)

In [ ]:
_stop_words
_stop_words.remove()

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [10]:
text = "I ate chocolate yesterday and I feel happy right now!"
preprocess_text(text)

'eat chocolate yesterday feel happy right'

In [7]:
df_balanced["clean_title"] = df_balanced["title"].apply(preprocess_text)

In [8]:
df_balanced[["title", "clean_title"]].head()

,title,clean_title
0,I haven't heard anything from my best/only fri...,hear anything friend day ever since ask could ...
1,"Gf advice needed plz Ok so, yesterday at the d...",gf advice need plz ok yesterday dance tell gir...
2,"Unpopular opinion: the word ""wholesome"" is gre...",unpopular opinion word wholesome great overuse...
3,I haven't left my house in a week On school ho...,leave house week school holiday go bed super l...
4,"I went to church today, for the lack of anythi...",go church today lack anything good really make...


In [9]:
df_balanced.drop(columns=["title"], inplace=True)

In [ ]:
#On sauvegarde en .csv
#df_balanced.to_csv("../../data/emotion_balanced.csv", index=False)

In [ ]:
#On sauvegarde la data en .pkl pour éviter de refaire le nettoyage à chaque fois
#df_balanced.to_pickle("../data/emotion_balanced.pkl")

In [48]:
# TODO: compute the BOW
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

# We create the output BOW, we can even reject directly the stop words and the punctuation, how magical?
vectorizer = CountVectorizer(max_features=10000,
                             stop_words='english',
                             #min_df=0.001, max_df=0.999
)
X = vectorizer.fit_transform(df_balanced["clean_title"])  # scipy sparse matrix
print(type(X), X.shape)  # should be (num_docs, num_features)

BOW = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=df_balanced.index,
    columns=vectorizer.get_feature_names_out(),
)

<class 'scipy.sparse._csr.csr_matrix'> (960818, 10000)


In [15]:
#On vérifie si BOW est bien une sparse matrix
sparse.issparse(BOW)

True

In [21]:
df_balanced["clean_title"][899]

'depress worthless end life would smart thing ive ever do awful person ive burden many people year jump job job look something didnt hate find doesnt exist give opportunity jump ship new excite place exactly job worthless hour week go dr bill dad support sure lift debt lose everything ive take much ive give contribute nothing world ive consume dont want try get back life hat want die deserve die beyond hopeless anymore feel like tomorrow think finally time shouldve do year ago dont want run away start anew dont like way world work fuck cant accept happiness need fight something inside u people tell worldviews cant comprehend hate world love family dont want around lose guess make selfish lazy doesnt surprise piece shit like bill burr say people simply dont deserve im one'

In [49]:
y = df_balanced["label"]
X.shape, y.shape

((960818, 10000), (960818,))

In [50]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.2, random_state=42,
                                                    stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((768654, 10000), (192164, 10000), (768654,), (192164,))

In [ ]:
# TODO: Perform a logistic regression to predict whether a message is a spam or not
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)
#91% pour preprocess et LogisticRegression de base.

0.9123405008222144

In [ ]:
######## ON REFAIT TOUT SANS BALANCE CLASSES 
######## ET AVEC CLASS_WEIGHT A LA FIN

In [5]:
df2 = load_data()
df2.shape

/home/thomas_d/code/Projet/mental-health-signal-detector/src/data_cleaning/data.py:54: DtypeWarning: Columns (0: Unnamed: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(_get_project_data_dir() / DATA_FILENAME)


(2470778, 8)

In [6]:
df2_cleaned = clean_data(df2)

In [7]:
df2_cleaned.shape

(2470669, 2)

In [8]:
sys.path.insert(0, os.path.abspath(".."))
from src.training.preprocess import preprocess_text, clean_text

In [10]:
#On applique clean_text et preprocess_text sur title
df2_cleaned["clean_title"] = df2_cleaned["title"].apply(clean_text)
df2_cleaned["clean_title"] = df2_cleaned["clean_title"].apply(preprocess_text)

In [11]:
df2_cleaned.drop(columns=["title"], inplace=True)

In [14]:
df2_cleaned.shape

(2470669, 2)

In [ ]:
#df2_cleaned.to_csv("../../data/emotion_preprocessed.csv", index=False)

In [16]:
# TODO: compute the BOW
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

# We create the output BOW, we can even reject directly the stop words and the punctuation, how magical?
vectorizer = CountVectorizer(max_features=10000,
                             stop_words='english',
                             #min_df=0.001, max_df=0.999
)
X = vectorizer.fit_transform(df2_cleaned["clean_title"])  # scipy sparse matrix
print(type(X), X.shape)  # should be (num_docs, num_features)

BOW = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=df2_cleaned.index,
    columns=vectorizer.get_feature_names_out(),
)

<class 'scipy.sparse._csr.csr_matrix'> (2470669, 10000)


In [17]:
y = df2_cleaned["label"]
X.shape, y.shape

((2470669, 10000), (2470669,))

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.2, random_state=42,
                                                    stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1976535, 10000), (494134, 10000), (1976535,), (494134,))

In [19]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000,class_weight="balanced")
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)

/home/thomas_d/.pyenv/versions/MHSD/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9355964171661938

In [23]:
from sklearn.metrics import classification_report
class_report = classification_report(y_test, log_reg.predict(X_test))

In [25]:
#Classification report en tableau
print(class_report)

              precision    recall  f1-score   support

         0.0       0.97      0.95      0.96    398052
         1.0       0.80      0.88      0.84     96082

    accuracy                           0.94    494134
   macro avg       0.89      0.92      0.90    494134
weighted avg       0.94      0.94      0.94    494134

